# Sprint 1 — Spaceship Titanic: EDA + Baseline Model

**Cel:** Załadować dane do SQLite, przeprowadzić EDA, wytrenować baseline model.


## 1. Importy i konfiguracja

In [ ]:
import sqlite3
import json
import os
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Ensure we run from the project root (portable — works on any OS)
_nb_dir = pathlib.Path().resolve()
if _nb_dir.name == 'notebooks':
    os.chdir(_nb_dir.parent)

load_dotenv()

DB_PATH = os.getenv('DATABASE_PATH', 'data/01_raw/dataset.db')
TRAIN_CSV = 'data/01_raw/train.csv'

plt.style.use('seaborn-v0_8')
%matplotlib inline

print('Libraries loaded ✓')
print(f'CWD: {os.getcwd()}')

## 2. Pobieranie danych z Kaggle

In [ ]:
!pip install -q kaggle

os.makedirs('data/01_raw', exist_ok=True)

if not os.path.exists(TRAIN_CSV):
    !kaggle competitions download -c spaceship-titanic -p data/01_raw
    !unzip -o data/01_raw/spaceship-titanic.zip -d data/01_raw
    print('Dane pobrane i rozpakowane ✓')
else:
    print('Dane już istnieją, pomijam pobieranie ✓')

## 3. Załaduj dane CSV → SQLite

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
print(f'Wczytano {len(df_raw)} wierszy, {len(df_raw.columns)} kolumn')

con = sqlite3.connect(DB_PATH)
df_raw.to_sql('spaceship_titanic', con, if_exists='replace', index=False)
con.close()
print(f'Dane zapisane do SQLite: {DB_PATH} ✓')

## 4. Wczytaj dane z SQLite

In [ ]:
con = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM spaceship_titanic', con)
con.close()

print(f'Załadowano z SQLite: {df.shape}')
df.head()

## 5. Statystyki opisowe

In [ ]:
df.describe(include='all')

In [ ]:
print('Typy kolumn:')
print(df.dtypes)
print(f'\nBraki danych:')
print(df.isnull().sum())
print(f'\nDuplikaty: {df.duplicated().sum()}')

## 6. Wizualizacje (EDA)

In [ ]:
# Wizualizacja 1: Rozkład zmiennej docelowej (Transported)
fig, ax = plt.subplots(figsize=(6, 4))
df['Transported'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Rozkład zmiennej docelowej: Transported')
ax.set_xlabel('Transported')
ax.set_ylabel('Liczba pasażerów')
ax.set_xticklabels(['False', 'True'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Wizualizacja 2: Rozkłady cech numerycznych
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), num_cols):
    df[col].dropna().hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Rozkłady cech numerycznych', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Wizualizacja 3: Heatmapa korelacji
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[num_cols + ['Transported']].copy()
corr['Transported'] = corr['Transported'].map({True: 1, False: 0})
sns.heatmap(corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Heatmapa korelacji cech numerycznych')
plt.tight_layout()
plt.show()

In [ ]:
# Wizualizacja 4: CryoSleep vs Transported
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='CryoSleep', hue='Transported', palette=['coral', 'steelblue'], ax=ax)
ax.set_title('CryoSleep vs Transported')
ax.set_xlabel('CryoSleep')
ax.set_ylabel('Liczba pasażerów')
plt.tight_layout()
plt.show()

In [ ]:
# Wizualizacja 5: Boxploty wydatków wg Transported
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df_spend = df[spend_cols + ['Transported']].copy()
df_spend['Transported'] = df_spend['Transported'].astype(str)
df_melt = df_spend.melt(id_vars='Transported', var_name='Usługa', value_name='Kwota')

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df_melt, x='Usługa', y='Kwota', hue='Transported',
            palette=['coral', 'steelblue'], showfliers=False, ax=ax)
ax.set_title('Wydatki wg kategorii i Transported (bez outlierów)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

### Viz 6: Inżynieria cechy Cabin (Deck i Side)

In [ ]:
# Cabin ma format: Deck/Num/Side  (np. 'B/0/P')
df_cabin = df.copy()
df_cabin[['Deck', 'Num', 'Side']] = df_cabin['Cabin'].str.split('/', expand=True)
df_cabin['Transported'] = df_cabin['Transported'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Deck vs Transported
deck_order = sorted(df_cabin['Deck'].dropna().unique())
sns.countplot(data=df_cabin, x='Deck', hue='Transported', order=deck_order,
              palette=['coral', 'steelblue'], ax=axes[0])
axes[0].set_title('Deck vs Transported')
axes[0].set_xlabel('Deck')
axes[0].set_ylabel('Liczba pasażerów')

# Side (Port/Starboard) vs Transported
sns.countplot(data=df_cabin, x='Side', hue='Transported',
              palette=['coral', 'steelblue'], ax=axes[1])
axes[1].set_title('Side (Port/Starboard) vs Transported')
axes[1].set_xlabel('Side')
axes[1].set_ylabel('Liczba pasażerów')

plt.tight_layout()
plt.show()

### Viz 7: VIP vs Transported

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='VIP', hue=df['Transported'].astype(str),
              palette=['coral', 'steelblue'], ax=ax)
ax.set_title('VIP vs Transported')
ax.set_xlabel('VIP')
ax.set_ylabel('Liczba pasażerów')
plt.tight_layout()
plt.show()

### Viz 8: Rozkład wieku wg Transported (KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for val, color, label in [(True, 'steelblue', 'Transported'), (False, 'coral', 'Not Transported')]:
    df[df['Transported'] == val]['Age'].dropna().plot.kde(
        ax=ax, color=color, label=label, linewidth=2)
ax.set_title('Rozkład wieku wg Transported (KDE)')
ax.set_xlabel('Wiek')
ax.set_ylabel('Gęstość')
ax.legend()
plt.tight_layout()
plt.show()

### Viz 9: Heatmapa braków danych

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False, ax=ax)
ax.set_title('Heatmapa braków danych (żółty = brak wartości)')
plt.tight_layout()
plt.show()
print(f'Łącznie brakujących wartości: {df.isnull().sum().sum()}')

## 7. Przygotowanie danych i podział train/val/test

In [ ]:
df_model = df.copy()

# Usuń kolumny tekstowe o wysokiej kardynalności
df_model = df_model.drop(columns=['PassengerId', 'Name', 'Cabin'])

# Enkodowanie zmiennych kategorycznych
cat_cols = df_model.select_dtypes(include=['object', 'bool']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'Transported']

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].astype(str)
    df_model[col] = le.fit_transform(df_model[col])

# Zmienna docelowa
df_model['Transported'] = df_model['Transported'].map({True: 1, False: 0, 'True': 1, 'False': 0})

# Uzupełnienie braków medianą
df_model = df_model.fillna(df_model.median(numeric_only=True))

X = df_model.drop(columns=['Transported'])
y = df_model['Transported']

# Podział: 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 8. Baseline model — RandomForest

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred_val = clf.predict(X_val)

metrics = {
    'accuracy': round(accuracy_score(y_val, y_pred_val), 4),
    'f1_score': round(f1_score(y_val, y_pred_val), 4)
}

print(f"Accuracy (val): {metrics['accuracy']}")
print(f"F1 Score (val): {metrics['f1_score']}")
print()
print(classification_report(y_val, y_pred_val))

### Viz 10: Ważność cech (Feature Importance)

In [ ]:
feat_imp = pd.Series(clf.feature_importances_, index=X_train.columns)
feat_imp_sorted = feat_imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp_sorted.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Ważność cech — RandomForest baseline')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Zapis metryk do JSON

In [ ]:
os.makedirs('data', exist_ok=True)
with open('data/metrics_sprint1.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Metryki zapisane do data/metrics_sprint1.json ✓')
print(json.dumps(metrics, indent=2))